In [16]:
from langchain_openai import ChatOpenAI
import os
from pydantic import SecretStr

llm = ChatOpenAI(
    model="gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=SecretStr(os.environ["OPENROUTER_API_KEY"]),
    temperature=0,
)   
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0', 'langchain-openai': '1.6.0'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000025087786DB0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000250878A77D0>, root_client=<openai.OpenAI object at 0x0000025087CC3DA0>

In [17]:
from langchain_core.messages import HumanMessage

llm.invoke([HumanMessage(content="Hello, how are you?")])

AIMessage(content="Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 13, 'total_tokens': 42, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 1.935e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 1.935e-05, 'upstream_inference_prompt_cost': 1.95e-06, 'upstream_inference_completions_cost': 1.74e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_ba6e3550a6', 'id': 'gen-1789641675-cqYei3k8fyjANAD15CJf', 'finish_reason': 'stop', 'logprobs': None}, id='lc_

In [18]:
from typing import TypedDict, List, Annotated
from operator import add

class graph_schema(TypedDict):

    messages_manual: List
    #Reducing Reducer
    messages_auto: Annotated[List,add]


In [19]:
from langchain_core.messages import AIMessage

def create_post(state: graph_schema) -> graph_schema:

    messages_manual = state['messages_manual']
    response_manual = llm.invoke(messages_manual).content 
    response__manual_ai = AIMessage(content=response_manual)
    state['messages_manual'] = messages_manual + [response__manual_ai]

    messages_auto = state['messages_auto']
    response_auto = llm.invoke(messages_auto).content
    response_auto_ai = AIMessage(content=response_auto)
    state['messages_auto'] = [response_auto_ai]

    return state

In [20]:

def curate_post(state: graph_schema) -> graph_schema:


    messages_manual = state['messages_manual']

    response_manual = llm.invoke(messages_manual).content
    response__manual_ai = AIMessage(content=response_manual)

    state['messages_manual'] = messages_manual + [response__manual_ai]

    messages_auto = state['messages_auto']
    response_auto = llm.invoke(messages_auto).content
    response_auto_ai = AIMessage(content=response_auto)
    state['messages_auto'] = [response_auto_ai]


    return state

In [21]:
from langgraph.graph import StateGraph,START,END

graph = StateGraph(graph_schema)

graph.add_node("create_post", create_post)
graph.add_node("curate_post", curate_post)

graph.add_edge(START, "create_post")
graph.add_edge("create_post", "curate_post")
graph.add_edge("curate_post", END)

messages_graph = graph.compile()

In [22]:
messages_graph.invoke(
    {"messages_manual": [HumanMessage(content="The beauty of nature")],
     "messages_auto": [HumanMessage(content="The beauty of nature")]}
)

{'messages_manual': [HumanMessage(content='The beauty of nature', additional_kwargs={}, response_metadata={}),
  AIMessage(content="The beauty of nature is a profound and multifaceted experience that captivates the senses and stirs the soul. It encompasses the vibrant colors of a sunset, the intricate patterns of leaves, the soothing sound of a flowing river, and the delicate fragrance of blooming flowers. Nature's beauty can be found in the grandeur of towering mountains, the tranquility of serene lakes, and the vastness of open fields.\n\nEach season brings its own unique charm: the fresh blooms of spring, the warmth of summer, the rich hues of autumn, and the serene stillness of winter. Wildlife adds another layer of beauty, from the graceful flight of birds to the playful antics of animals in their natural habitats.\n\nBeyond its aesthetic appeal, nature also offers a sense of peace and connection. Many find solace in the great outdoors, where the hustle and bustle of daily life fa